<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/NER_COMBINED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

תא 1 — התקנות (Python)

In [1]:
%pip install -q "transformers==4.49.0" "spacy>=3.8,<3.9" "spacy-transformers>=1.3,<1.4" "gliner2[local]==2.1.0" "flair==0.15.1" "huggingface-hub>=0.30,<1" "packaging>=24,<27"
!python -m spacy download en_core_web_trf
!python -m pip check

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 968.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.8/355.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.6/779.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

תא 2 — כל הייבואים (Python)

In [3]:
import gc
import hashlib
import json
import logging
import platform
import random
import re
import subprocess
import time
import uuid
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import spacy
import torch
import transformers
from flair.models import SequenceTagger
from flair.splitter import SegtokSentenceSplitter
from gliner2 import AutoExtractor
from google.colab import files
from huggingface_hub import hf_hub_download, model_info, snapshot_download
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("colab_ner")
print ("כל הספריות נטענו")

כל הספריות נטענו


תא 3 — הגדרות מרכזיות (Python)

In [4]:
SAMPLE_SIZE = 200
SEED = 93

# הנתיב המקורי מהמחברת. אפשר להצביע במקום זאת ישירות על DATA.json קיים ב-Colab.
REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
REPO_DIR = Path("/content/Corpus2GeoRAG_Pacific")
SOURCE_PATH = None  # למשל Path("/content/DATA.json")

# נתיבים לקובצי sample_*.json קודמים, אם רוצים לדווח חפיפה.
PREVIOUS_SAMPLE_PATHS = []

HF_STRIDE = 64
GLINER_THRESHOLD = 0.5
GLINER_CHUNK_SIZE = 384
GLINER_CHUNK_OVERLAP = 64
HIGH_SINGLETON_SCORE = 0.99

BERT_ID = "dslim/bert-base-NER"
BERT_REVISION = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"
ROBERTA_ID = "Jean-Baptiste/roberta-large-ner-english"
SPACY_ID = "en_core_web_trf"
GLINER_ID = "fastino/gliner2.5-base-v1"
FLAIR_ID = "flair/ner-english-ontonotes-large"

GLINER_LABELS = [
    "person", "organization", "military unit", "location",
    "geopolitical entity", "facility", "nationality or political group",
    "event", "date", "time", "weapon", "vehicle", "product",
    "work of art", "law", "language", "money", "quantity",
    "percent", "ordinal", "cardinal", "miscellaneous named entity",
]

assert SAMPLE_SIZE > 0 and isinstance(SEED, int)
assert len(GLINER_LABELS) == 22
assert 0 <= GLINER_CHUNK_OVERLAP < GLINER_CHUNK_SIZE

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = Path("/content/TMP_NER") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
ZIP_PATH = Path("/content") / f"ner_run_{RUN_ID}.zip"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

def save_json(path, obj):
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(obj, handle, ensure_ascii=False, indent=2)

def jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def package_versions():
    names = ["torch", "transformers", "spacy", "spacy-transformers",
             "gliner2", "flair", "huggingface-hub"]
    return {name: version(name) for name in names}

PACKAGE_VERSIONS = package_versions()
MODEL_FILES = {}
print("RUN_ID:", RUN_ID)
print("ספריות:", PACKAGE_VERSIONS)
print("GPU:", torch.cuda.is_available())

RUN_ID: 20261011T150155Z_14e71336
ספריות: {'torch': '2.11.0+cpu', 'transformers': '4.49.0', 'spacy': '3.8.16', 'spacy-transformers': '1.3.9', 'gliner2': '2.1.0', 'flair': '0.15.1', 'huggingface-hub': '0.36.2'}
GPU: False
